<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Execution Tracking </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Demonstrate how to add **execution tracking** to a multi-agent system so we can monitor progress, update task status, and detect repeated or failed steps.

**New Scenario:** AI Research Report Generator  
We will build a system that generates a short research report on "Latest advancements in Multimodal AI models in 2026".

**Key Topics Covered:**
- Task status tracking (Pending, In Progress, Completed, Failed)
- Maintaining execution state
- Detecting repeated or stalled steps
- Logging and visualizing progress across steps

---

## 1. Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" duckduckgo-search --force-reinstall
!pip install -U ddgs

import os
from getpass import getpass
from typing import Dict, List, Optional
from enum import Enum

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2

In [2]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client

client = Client()

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform mathematical calculations."""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Tools and LLM ready")

Tools and LLM ready


## 2. Task Status Tracking System

In [3]:
class TaskStatus(Enum):
    PENDING = "Pending"
    IN_PROGRESS = "In Progress"
    COMPLETED = "Completed"
    FAILED = "Failed"

class Task:
    def __init__(self, id: int, description: str):
        self.id = id
        self.description = description
        self.status = TaskStatus.PENDING
        self.result = None
        self.error = None

    def update(self, status: TaskStatus, result: Optional[str] = None, error: Optional[str] = None):
        self.status = status
        if result:
            self.result = result
        if error:
            self.error = error

    def __repr__(self):
        return f"Task {self.id}: {self.description} → {self.status.value}"

# Execution Tracker
class ExecutionTracker:
    def __init__(self):
        self.tasks: List[Task] = []
        self.current_step = 0

    def add_task(self, description: str):
        task = Task(len(self.tasks) + 1, description)
        self.tasks.append(task)
        return task

    def mark_in_progress(self, task_id: int):
        for task in self.tasks:
            if task.id == task_id:
                task.update(TaskStatus.IN_PROGRESS)
                self.current_step = task_id
                print(f"Starting Task {task_id}: {task.description}")
                return

    def mark_completed(self, task_id: int, result: str):
        for task in self.tasks:
            if task.id == task_id:
                task.update(TaskStatus.COMPLETED, result=result)
                print(f"Completed Task {task_id}")
                return

    def mark_failed(self, task_id: int, error: str):
        for task in self.tasks:
            if task.id == task_id:
                task.update(TaskStatus.FAILED, error=error)
                print(f"Failed Task {task_id}: {error}")
                return

    def print_progress(self):
        print("\nExecution Progress:")
        for task in self.tasks:
            print(f"   {task}")
        print("-" * 60)

print("TaskStatus and ExecutionTracker created")

TaskStatus and ExecutionTracker created


## 3. Worker Agents with Tracking

In [4]:
# Simple Researcher and Calculator with tracking support
researcher_executor = AgentExecutor(
    agent=create_react_agent(llm=llm, tools=[search_tool], prompt=client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt= True)),
    tools=[search_tool],
    verbose=False,
    max_iterations=6,
    handle_parsing_errors=True
)

calculator_executor = AgentExecutor(
    agent=create_react_agent(llm=llm, tools=[calculate_math], prompt=client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt= True)),
    tools=[calculate_math],
    verbose=False,
    max_iterations=6,
    handle_parsing_errors=True
)

print("Worker Agents ready")

Worker Agents ready


## 4. Demo: Execution Tracking in Action

In [5]:
def run_execution_tracking_demo():
    tracker = ExecutionTracker()

    print("Starting AI Research Report Generator with Execution Tracking\n")

    # Create tasks
    tracker.add_task("Research latest advancements in multimodal AI models (2025-2026)")
    tracker.add_task("Identify the top 2 key breakthroughs or models")
    tracker.add_task("Summarize the main modalities supported by these models")
    tracker.add_task("Calculate the average number of modalities across the top models")
    tracker.add_task("Compile a clean final research summary")

    tracker.print_progress()
    print("\n" + "="*70 + "\n")

    results = {}  # Store actual results for final summary

    try:
        # Task 1: Research Advancements
        tracker.mark_in_progress(1)
        research_data = researcher_executor.invoke({
            "input": "Latest advancements and new multimodal AI models in 2025-2026"
        })["output"]
        tracker.mark_completed(1, "Research completed")
        results["research"] = research_data
        print("Research Output:\n", research_data[:500] + "..." if len(research_data) > 500 else research_data, "\n")

        # Task 2: Top 2 Models
        tracker.mark_in_progress(2)
        top_models = researcher_executor.invoke({
            "input": "Identify the top 2 multimodal AI models or breakthroughs in 2025-2026"
        })["output"]
        tracker.mark_completed(2, "Top models identified")
        results["top_models"] = top_models
        print("Top Models:\n", top_models[:400] + "..." if len(top_models) > 400 else top_models, "\n")

        # Task 3: Modalities
        tracker.mark_in_progress(3)
        modalities = researcher_executor.invoke({
            "input": "List the main modalities supported by the top multimodal AI models (text, image, video, audio, etc.)"
        })["output"]
        tracker.mark_completed(3, "Modalities summarized")
        results["modalities"] = modalities

        # Task 4: Average Calculation
        tracker.mark_in_progress(4)
        calc_input = "Assume the top 2 multimodal models support 4 and 3 modalities respectively. Calculate the average number of modalities."
        calc_result = calculator_executor.invoke({"input": calc_input})["output"]
        tracker.mark_completed(4, f"Average modalities: {calc_result}")
        results["average"] = calc_result

        # Task 5: Final Summary
        tracker.mark_in_progress(5)
        final_summary = f"""Final Research Summary on Multimodal AI (2025-2026):

Key Advancements: {results.get('research', 'N/A')[:300]}...

Top Models: {results.get('top_models', 'N/A')[:200]}...

Average Modalities: {results.get('average', 'N/A')}

Conclusion: Multimodal AI continues to advance rapidly with increasing support for multiple data types."""

        tracker.mark_completed(5, "Final summary generated")
        print("Final Summary:\n", final_summary)

    except Exception as e:
        tracker.mark_failed(tracker.current_step, str(e))

    print("\n" + "="*70)
    tracker.print_progress()

    print("\nExecution Tracking Demo Completed!")
    print("We can now see both the progress AND the actual results of each step.")

run_execution_tracking_demo()

Starting AI Research Report Generator with Execution Tracking


Execution Progress:
   Task 1: Research latest advancements in multimodal AI models (2025-2026) → Pending
   Task 2: Identify the top 2 key breakthroughs or models → Pending
   Task 3: Summarize the main modalities supported by these models → Pending
   Task 4: Calculate the average number of modalities across the top models → Pending
   Task 5: Compile a clean final research summary → Pending
------------------------------------------------------------


Starting Task 1: Research latest advancements in multimodal AI models (2025-2026)
Completed Task 1
Research Output:
 In 2025-2026, significant advancements in multimodal AI models include the launch of Google's Gemma 4 12B model, which integrates audio, visual understanding, and multi-step reasoning for local deployment. Additionally, DeepSeek V4 has reached 1 trillion total parameters with enhanced efficiency. The field is seeing a trend towards smaller, more efficient m

## 5. Reflection Questions

1. How does the Planning Agent help prevent infinite loops?
2. What are the benefits of generating a structured plan before execution?
3. How did we delegate tasks to specialized worker agents?
4. What challenges might occur if the plan is too vague or too long?
5. How would you improve this system to make delegation more dynamic?
